In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fstreet_view_insights%2Ffull_frame%2Fmulti_observation_asset_analysis.ipynb?utm_source=full_frame_street_view_insights_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
</table>

# Multi-Observation Asset Analysis with Gemini 3.5 Flash

This notebook demonstrates how to group Street View observation images by `asset_id` and query Gemini 3.5 Flash with multiple images simultaneously. This enables a unified, high-confidence visual analysis of a single asset across all its captured perspectives while calculating the total processing cost.

## Install Required Libraries

In [ ]:
!pip install --upgrade google-cloud-bigquery google-genai google-cloud-storage "pillow<11.0.0" matplotlib

## Configuration

**Important**: Replace the placeholder values below with your actual GCP Project ID and Region.

In [ ]:
PROJECT_ID = 'YOUR_PROJECT_ID'  # @param {type:"string"}
REGION = 'global'      # @param {type:"string"}

# BigQuery Configuration
BIGQUERY_DATASET_ID = 'imagery_insights___us' # @param {type:"string"}
BIGQUERY_TABLE_ID = 'full_frame_observations_latest' # @param {type:"string"}
ASSET_LIMIT = 3 # @param {type:"integer"}
ASSET_TYPE = "ASSET_CLASS_UTILITY_POLE" # @param {type:"string"}
MODEL = "gemini-3.5-flash" # @param {type:"string"}
THINKING_LEVEL = "HIGH" # @param ["MINIMAL", "LOW", "MEDIUM", "HIGH"] {type:"string"}

## Imports and SDK Initialization

In [ ]:
import io
import vertexai
import PIL.Image
import PIL.ImageDraw
import matplotlib.pyplot as plt
from google.cloud import bigquery
from google.cloud import storage
from google import genai
from google.genai import types
from google.genai.types import Content, Part

# Initialize Vertex AI SDK and Gemini Client
vertexai.init(project=PROJECT_ID, location=REGION)
client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

## Fetch Asset IDs and Observations from BigQuery

We query BigQuery to group observations of the target asset class by `asset_id` using `ARRAY_AGG`.

In [ ]:
BIGQUERY_SQL_QUERY = f"""
SELECT
  asset_id,
  ARRAY_AGG(STRUCT(gcs_uri, bbox, detection_time, pano_id)) as observations
FROM
  `{PROJECT_ID}.{BIGQUERY_DATASET_ID}.{BIGQUERY_TABLE_ID}`
WHERE asset_type = '{ASSET_TYPE}'
GROUP BY asset_id
LIMIT {ASSET_LIMIT};
"""

# Execute BigQuery Query
try:
    bigquery_client = bigquery.Client(project=PROJECT_ID)
    query_job = bigquery_client.query(BIGQUERY_SQL_QUERY)
    
    assets = []
    for row in query_job:
        obs_list = []
        for obs in row.get("observations", []):
            if obs.get("gcs_uri"):
                obs_list.append({
                    "gcs_uri": obs.get("gcs_uri"),
                    "bbox": obs.get("bbox")
                })
        if obs_list:
            assets.append({
                "asset_id": row.get("asset_id"),
                "observations": obs_list
            })

    print(f"Successfully fetched {len(assets)} assets with their observations.")
    for asset in assets:
        print(f"Asset ID: {asset['asset_id']} | Observations: {len(asset['observations'])}")
except Exception as e:
    print(f"An error occurred while querying BigQuery: {e}")

## Bounding Box Visualization Helper

Define helpers to load images and draw bounding boxes for each of the views of the asset.

In [ ]:
def display_image_with_bbox(gcs_uri: str, bbox: dict, label: str = None):
    """
    Downloads the image from GCS, draws the bounding box, and displays it.
    """
    try:
        if not gcs_uri.startswith("gs://"):
            print("Invalid GCS URI")
            return
            
        parts = gcs_uri[5:].split("/", 1)
        bucket_name = parts[0]
        blob_name = parts[1]
        
        storage_client = storage.Client(project=PROJECT_ID)
        bucket = storage_client.bucket(bucket_name)
        blob = bucket.blob(blob_name)
        image_bytes = blob.download_as_bytes()
        
        image = PIL.Image.open(io.BytesIO(image_bytes))
        draw = PIL.ImageDraw.Draw(image)
        
        # BQ bbox coordinates
        xmin = bbox['lo']['x']
        ymin = bbox['lo']['y']
        xmax = bbox['hi']['x']
        ymax = bbox['hi']['y']
        
        # Draw bounding box
        draw.rectangle([xmin, ymin, xmax, ymax], outline="red", width=10)
        
        if label:
            draw.text((xmin + 20, ymin + 20), label, fill="red")
            
        plt.figure(figsize=(12, 8))
        plt.imshow(image)
        plt.axis('off')
        plt.show()
    except Exception as e:
        print(f"Error displaying image: {e}")

def display_asset_observations(asset_id: str, observations: list):
    """
    Displays all observations for a given asset ID with bounding boxes drawn.
    """
    print(f"Asset ID: {asset_id} has {len(observations)} observations.")
    for i, obs in enumerate(observations):
        uri = obs["gcs_uri"]
        bbox = obs["bbox"]
        print(f"Observation {i+1}: {uri}")
        display_image_with_bbox(uri, bbox, label=f"Obs {i+1}")

## Define Combined Image Analysis Function

This function accepts a list of observation objects, appends all of their GCS URIs as image parts to a single content list, executes the Gemini API request, and calculates the total cost of the multimodal inference call.

In [ ]:
def analyze_asset_with_gemini(asset_id: str, observations: list, prompt: str) -> tuple[str, float, int, int]:
    """
    Passes all observation images of the asset combined to Gemini for unified understanding.
    Returns response text, calculated cost, prompt token count, and candidates token count.
    """
    try:
        contents = [prompt]
        
        # Append all images
        for obs in observations:
            gcs_uri = obs["gcs_uri"]
            contents.append(Part(file_data={'file_uri': gcs_uri, 'mime_type': 'image/jpeg'}))
            
        # Configure thinking config
        config = types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(
                thinking_level=THINKING_LEVEL
            )
        )
        
        response = client.models.generate_content(model=MODEL, contents=contents, config=config)
        
        # Calculate cost dynamically from usage metadata
        prompt_tokens = response.usage_metadata.prompt_token_count
        completion_tokens = response.usage_metadata.candidates_token_count
        
        # Pricing for gemini-3.5-flash: Input: $0.000075 / 1k, Output: $0.00030 / 1k
        input_cost = prompt_tokens * (0.000075 / 1000)
        output_cost = completion_tokens * (0.00030 / 1000)
        total_cost = input_cost + output_cost
        
        return response.text, total_cost, prompt_tokens, completion_tokens
    except Exception as e:
        print(f"Error analyzing asset {asset_id}: {e}")
        return "Analysis failed.", 0.0, 0, 0

## Perform Multi-Observation Analysis

Finally, iterate over all retrieved assets, drawing their bounding boxes, running the unified multi-image prompt, and calculating the combined GCP API costs.

In [ ]:
prompt = """You are provided with multiple photos showing different views of the exact same utility pole:
{photos_of_utility_pole}

Instructions:
1. Analyze all the provided images together. They show the same object from different angles and distances. Combine the visual information to provide a single, unified, high-confidence report.
2. Detect and count the following across the entire asset:
    * Transformers
    * Power lines coming from the pole
    * Street lamps attached to the pole
    * Telephone or junction boxes
3. Assess the overall condition of the pole. Look for visible damage, bird nests, or structural issues. If it appears in good condition, note \"OK\".
4. Note the material (e.g., wood, metal, concrete) the pole is made of.
5. Determine the primary type of the pole (Street light, High tension power transmission, electricity pole, or other).
6. Return your findings in the following JSON format:

```json
{
  \"pole_condition\": \"OK/Damaged/Other Issues\",
  \"type\": \"<pole_type>\",
  \"material\": \"<material>\",
  \"transformers\": <number_of_transformers>,
  \"power_lines\": <number_of_power_lines>,
  \"street_lamps\": <number_of_street_lamps>,
  \"junction_boxes\": <number_of_junction_boxes>,
  \"additional_notes\": \"<summary of observations combined across all images>\"
}
```
"""

if 'assets' in locals() and assets:
    total_run_cost = 0.0
    for item in assets:
        asset_id = item["asset_id"]
        observations = item["observations"]
        
        print(f"\n==================================================")
        print(f"Analyzing Asset: {asset_id}")
        print(f"==================================================")
        
        # Draw bounding boxes for all views
        display_asset_observations(asset_id, observations)
        
        # Run combined Gemini analysis
        print("\nRunning combined Gemini multi-image analysis...")
        analysis_result, cost, in_tokens, out_tokens = analyze_asset_with_gemini(asset_id, observations, prompt)
        
        print(f"Combined Result:\n{analysis_result}")
        print(f"Tokens - Input: {in_tokens} | Output: {out_tokens}")
        print(f"Gemini API cost for this asset (all observations): ${cost:.6f}")
        total_run_cost += cost
        
    print(f"\n==================================================")
    print(f"Total API Cost for run: ${total_run_cost:.6f}")
    print(f"==================================================")
else:
    print("No assets found to analyze.")